In [1]:

import sys, os
sys.path.append(os.path.abspath("..")) 
import json
import torch
from datasets import CNFDataset
from models import LightningModelCNF
from utils import args_cnf, gen_image_cnf, plotly_generate
import random
import itertools
import numpy as np
import matplotlib.pyplot as plt

**Read configuration files and arguments:**

In [3]:
# Arguments
parser = args_cnf()
args, unknown = parser.parse_known_args()

args.particle = "proton_contained"
args.metadata_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/metadata.pkl"
args.dataset_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/{}/{}/{}/{}.zip"
args.cnf_ind_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/gan_ind.pkl"
args.save_dir = "/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/test_spline_noexiting_rotation/"
args.checkpoint_path = "/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/test_spline_noexiting_rotation/checkpoints_v2"
args.checkpoint_name = args.particle

if args.particle == "muon" or args.particle == "proton_exiting":
    args.label_size = 7
elif args.particle == "proton_contained":
    args.label_size = 7
args.epochs = 50
args.log_every_n_steps = 2000
args.batch_size = 512
args.hidden = 256
args.num_workers = 64

def apply_rotation_or_flip(hit_x, hit_y, hit_z, pos_ini_mod, inidir):
        """
        Apply rotation or flip to the image.
        """
        hit_x = hit_x - 2
        hit_y = hit_y - 2
        hit_z = hit_z - 2
        # rot_flip matrix
        perm = random.choice(list(itertools.permutations([0, 1, 2])))
        #perm = [1, 0, 2]
        signs = [random.choice([-1, 1]) for _ in range(3)]
        #signs = [1, -1, 1]

        R = np.zeros((3, 3), dtype=np.float32)
        for i, p in enumerate(perm):
            R[i, p] = float(signs[i])
        
        xyz = np.stack([hit_x, hit_y, hit_z], axis=1).astype(np.float32)
        xyz = xyz @ R.T
        xyz = xyz.astype(np.int32)
        
        hit_x, hit_y, hit_z = xyz[:,0], xyz[:,1], xyz[:,2]
        inidir = R @ inidir
        pos_ini_mod = R @ pos_ini_mod

        hit_x = hit_x + 2
        hit_y = hit_y + 2
        hit_z = hit_z + 2

        return hit_x, hit_y, hit_z, pos_ini_mod, inidir

def get_xyzq_from_image(image):
    """
    Get the x, y, z, and q values from the image.
    """
    x, y, z= np.where(image.reshape(5, 5, 5) > 0)
    q = image.reshape(5, 5, 5)[x, y, z]
    return x, y, z, q

def draw_3d_image(x,y,z,q):
    """
    Draw the 3D image.
    """
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(x, y, z, c=q, cmap='viridis')
    plt.show()

In [4]:
test_set_p = CNFDataset(args, split="val")
event = test_set_p[1]
ini_pos = event['pos_ini'].numpy()
ini_dir = event['dir_ini'].numpy()
max_charge = np.log(test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max'] + 1)
image = (event['image'].numpy()+1)/2
image *= (max_charge - 0)
image = np.exp(image) - 1
hit_x, hit_y, hit_z, hit_q = get_xyzq_from_image(image)
print(hit_x, hit_y, hit_z, hit_q)
print(test_set_p.__len__())
print(ini_pos, ini_dir)


[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4] [0 0 0 0 0 1 1 1 1 1 2 2 2 2 2 3 3 3 3 3 4 4 4 4 4 0 0 0 0 0 1 1 1 1 1 2 2
 2 2 2 3 3 3 3 3 4 4 4 4 4 0 0 0 0 0 1 1 1 1 1 2 2 2 2 2 3 3 3 3 3 4 4 4 4
 4 0 0 0 0 0 1 1 1 1 1 2 2 2 2 2 3 3 3 3 3 4 4 4 4 4 0 0 0 0 0 1 1 1 1 1 2
 2 2 2 2 3 3 3 3 3 4 4 4 4 4] [0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1
 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3
 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4 0
 1 2 3 4 0 1 2 3 4 0 1 2 3 4] [4.26286398e-01 6.77500411e-01 2.36255917e-01 2.68090122e-01
 1.52330755e+00 1.60585952e+00 8.73782388e-02 3.37195319e-02
 7.59731605e-02 6.68355131e-01 1.29479851e+00 9.12542156e-02
 1.74301264e-01 4.79000262e-01 1.30095061e-01 1.2350

125000


In [5]:
image_3d = image.reshape(5, 5, 5)
print(image)
print(image.max())

[4.26286398e-01 6.77500411e-01 2.36255917e-01 2.68090122e-01
 1.52330755e+00 1.60585952e+00 8.73782388e-02 3.37195319e-02
 7.59731605e-02 6.68355131e-01 1.29479851e+00 9.12542156e-02
 1.74301264e-01 4.79000262e-01 1.30095061e-01 1.23504700e-01
 1.46537692e-01 6.96582774e-01 1.15380633e+00 7.26726959e-01
 2.10183454e-01 4.15292330e-01 5.31250976e-01 2.65154153e-01
 7.75657017e-01 1.50790471e+00 1.29680766e-01 6.58963269e-01
 2.93291703e-01 1.07668124e+00 1.35270641e-01 1.17222022e+00
 8.72788959e-01 1.35965030e+00 7.59477892e-01 6.64786270e-01
 8.78520411e-01 5.55769194e-01 1.26968054e+00 1.41527049e+00
 3.71396851e-01 1.11532191e-02 3.62485364e-01 6.47235365e-01
 5.90298387e-01 2.46101630e-01 3.83113065e-01 1.47358383e+00
 9.08265040e-01 4.19028227e-01 1.60249373e+00 7.08228732e-01
 1.44206073e+00 5.61719798e-02 9.56673987e-01 6.14829536e-01
 3.42790234e-01 2.19115904e-01 7.68816644e-01 1.28091370e-01
 1.63642518e+00 1.79560393e-01 2.50239045e+02 2.69013389e-01
 5.00823298e-01 1.270452

In [6]:
plotly_generate(image.reshape(5, 5, 5), max_energy=image.max()+1)

1229
1
1
1
1
1
1
1
1
1
1
1
1
1
1
250
34
272
64
1
4
46
1
1
1
1
1
1
99
1
1
365
619
30
104
1227
78
1
1
1
1
1
1
23
19
1
4
102


In [48]:
x_mod, y_mod, z_mod, ini_pos_mod, ini_dir_mod = apply_rotation_or_flip(hit_x, hit_y, hit_z, ini_pos, ini_dir)

In [49]:
print(x_mod, y_mod, z_mod)
print(hit_x, hit_y, hit_z)

[4 3 3 4 4 2 3 3] [4 3 3 3 3 2 2 2] [4 2 3 3 4 2 2 3]
[0 1 1 1 1 2 2 2] [4 3 3 4 4 2 3 3] [4 2 3 3 4 2 2 3]


In [50]:
print(ini_pos_mod, ini_dir_mod)
print(ini_pos, ini_dir)

[ 0.16583307 -0.17157799 -0.04362139] [0.5790125 0.5950701 0.5573475]
[ 0.17157799  0.16583307 -0.04362139] [-0.5950701  0.5790125  0.5573475]


In [51]:
image_mode = np.zeros((5, 5, 5))
image_mode[x_mod, y_mod, z_mod] = hit_q
plotly_generate(image_mode, max_energy=image_mode.max()+1)


142
37
88
46
19
115
94
19
140


**Load the pre-trained weights of the different generative-adversarial-network (GAN) models:**

In [7]:
# Dataset and generator models
test_set_p = CNFDataset(args, split="val")

checkpoint_path = "/".join((args.checkpoint_path, args.checkpoint_name, "train_loss", "last.ckpt"))


model = LightningModelCNF.load_from_checkpoint(checkpoint_path, img_shape = (args.img_size, args.img_size, args.img_size), 
                                    label_size = args.label_size, 
                                    hidden_features = args.hidden, 
                                    num_blocks_in_MADE = args.num_blocks_in_MADE, 
                                    num_transformers = args.num_transformers, 
                                    lr = args.lr, 
                                    wd = args.weight_decay)

model.eval()

# move to cpu
model.to("cpu")


LightningModelCNF(
  (nflow): Flow(
    (_transform): CompositeTransform(
      (_transforms): ModuleList(
        (0): MaskedPiecewiseRationalQuadraticAutoregressiveTransform(
          (autoregressive_net): MADE(
            (initial_layer): MaskedLinear(in_features=125, out_features=256, bias=True)
            (context_layer): Linear(in_features=7, out_features=256, bias=True)
            (activation): ReLU()
            (blocks): ModuleList(
              (0-1): 2 x MaskedResidualBlock(
                (context_layer): Linear(in_features=7, out_features=256, bias=True)
                (linear_layers): ModuleList(
                  (0-1): 2 x MaskedLinear(in_features=256, out_features=256, bias=True)
                )
                (activation): ReLU()
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (final_layer): MaskedLinear(in_features=256, out_features=3625, bias=True)
          )
        )
        (1): BatchNorm()
        (2)

In [8]:
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def generate_image(labels:torch.Tensor, model:LightningModelCNF, n_samples:int=1, min_charge:float=0, max_charge:float=None):
    generated_p = model.sample(labels, num_samples=n_samples)
    generated_p = generated_p.numpy()
    generated_p = generated_p.reshape(n_samples, 5, 5, 5)

    generated_p = (generated_p + 1) / 2
    generated_p *= (max_charge - min_charge)
    generated_p += min_charge
    generated_p = np.exp(generated_p) - 1
    generated_p[generated_p < 0] = 0
    return generated_p

def find_best_image(x:torch.Tensor, x_generated:torch.Tensor):
    # find the image with the highest cosine similarity to x
    losses = []
    # softmax the images
    x_generated = F.log_softmax(x_generated, dim=-1)
    x = F.log_softmax(x, dim=-1)
    for i in range(x_generated.shape[0]):
        # calculate the kv divergence plus l2 difference
        kl_divergence = F.kl_div(x_generated[i], x)
        l2_difference = F.mse_loss(x_generated[i], x)
        losses.append(kl_divergence + l2_difference)
    return np.argmin(losses), x_generated[np.argmin(losses)]

def grid_to_sparse_points(grid, threshold=1e-6):
    """
    grid: numpy array (D, H, W)
    Returns arrays x, y, z, val for voxels above |value| > threshold.
    """

    mask = np.abs(grid) > threshold
    z_idx, y_idx, x_idx = mask.nonzero()  # (z, y, x)
    vals = grid[mask]
    return x_idx, y_idx, z_idx, vals

def make_cube_traces(x, y, z, v, cmin, cmax, showscale_first=True):
    """
    Build a list of Mesh3d traces, one per (x, y, z) cube.
    Opacity is proportional to v/cmax.
    Each cube has hover text with its coordinates and value.
    """

    traces = []
    first = True

    for cx, cy, cz, val in zip(x, y, z, v):
        # 8 corners of cube [cx, cx+1] x [cy, cy+1] x [cz, cz+1]
        verts = np.array([
            (cx,   cy,   cz),
            (cx+1, cy,   cz),
            (cx+1, cy+1, cz),
            (cx,   cy+1, cz),
            (cx,   cy,   cz+1),
            (cx+1, cy,   cz+1),
            (cx+1, cy+1, cz+1),
            (cx,   cy+1, cz+1),
        ])
        xs, ys, zs = verts[:, 0], verts[:, 1], verts[:, 2]

        # 12 triangles (2 per face)
        faces = [
            (0, 1, 2), (0, 2, 3),  # bottom
            (4, 5, 6), (4, 6, 7),  # top
            (0, 1, 5), (0, 5, 4),  # front
            (2, 3, 7), (2, 7, 6),  # back
            (1, 2, 6), (1, 6, 5),  # right
            (0, 3, 7), (0, 7, 4),  # left
        ]
        i = [f[0] for f in faces]
        j = [f[1] for f in faces]
        k = [f[2] for f in faces]

        intensity = [val] * 8
        opacity = 0.0 if cmax <= 0 else float(val) / cmax

        hover_text = [f"x={cx}, y={cy}, z={cz}, value={val:.4g}"] * 8

        traces.append(
            go.Mesh3d(
                x=xs,
                y=ys,
                z=zs,
                i=i,
                j=j,
                k=k,
                intensity=intensity,
                intensitymode="vertex",
                colorscale="Viridis",
                cmin=cmin,
                cmax=cmax,
                opacity=opacity,
                flatshading=True,
                text=hover_text,
                hoverinfo="text",
                showscale=showscale_first and first,
                colorbar=dict(title="Energy") if (showscale_first and first) else None,
            )
        )
        if first:
            first = False

    return traces

def draw_data_vs_generated(hit_charge, sample, output_plot_name):

    data = hit_charge
    generated = sample
    generated[generated<0.5] = 0
    global_max = float(max(data.max(), generated.max()))
    cmin, cmax = 0.01, global_max if global_max > 0 else 1.0
    x_t, y_t, z_t, v_t = grid_to_sparse_points(data)
    x_g, y_g, z_g, v_g = grid_to_sparse_points(generated)

    true_traces = make_cube_traces(x_t, y_t, z_t, v_t, cmin, cmax, showscale_first=True)
    gen_traces  = make_cube_traces(x_g, y_g, z_g, v_g, cmin, cmax, showscale_first=True)

    # Plot
    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=("True", "Generated"),
    )
    for tr in true_traces:
        fig.add_trace(tr, row=1, col=1)
    for tr in gen_traces:
        fig.add_trace(tr, row=1, col=2)
    axis_range = [-0.5, 5.5]
    for col in [1, 2]:
        fig.update_scenes(
            xaxis=dict(title="x", range=axis_range),
            yaxis=dict(title="y", range=axis_range),
            zaxis=dict(title="z", range=axis_range),
            aspectmode="cube",
            row=1,
            col=col,
        )

    fig.update_layout(
        showlegend=False,
    )

    # save the figure
    fig.write_image(f"{output_plot_name}.png")
    #plt.close()
    return

**Run each GAN on some arbitrary input kinematics:**

In [ ]:
'''
Proton GAN
'''
import numpy as np

# Set your kinematics here:
# ke = 30.3  # Initial kinetic energy
# ini_dir = [0.9999999999999999, 0.0, 0.0]  # Initial direction
# ini_pos = [-1.5, -4.2, 2.7]  # Initial 3D position (mm)

# get n random event from the test set
n=1
n_samples = 10
rand_indices = np.random.randint(0, len(test_set_p), n)
min_charge = 0
max_charge = np.log(test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max'] + 1)
output_folder = f"/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/test_spline_noexiting_rotation/plots/{args.particle}/best_fit_images"
for i in rand_indices:
    event = test_set_p[0]
    # convert torch tensor to numpy array
    ke = float(event['ke'].numpy())
    ini_pos = event['pos_ini'].numpy()
    ini_dir = event['dir_ini'].numpy()

    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2]])

#params = np.array([0.1925,  0.2249, -0.0153, -0.1201, -0.4700, -0.4900, -0.7400])
    labels = torch.tensor([params], dtype=torch.float32)

    print(labels)

    generated_p = generate_image(labels, model, n_samples, min_charge, max_charge)
    img = event['image']
    img = (img + 1) / 2
    img *= (max_charge - min_charge)
    img += min_charge
    img = np.exp(img) - 1

    print(generated_p.shape)
    
    #best_index, best_image = find_best_image(img, generated_p)

    #max_energy = max(img.max(), best_image.max())

    #plot
    #draw_data_vs_generated(img, best_image, f"{output_folder}/data_vs_generated_{i}")
    break



/tmp/ipykernel_1260427/1405874126.py:21: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/tmp/ipykernel_1260427/1405874126.py:28: UserWarning:

Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1758491028874/work/torch/csrc/utils/tensor_new.cpp:253.)



tensor([[-0.1965, -0.3004,  0.0163,  0.1135,  0.1274,  0.9575, -0.2588]])


(10, 5, 5, 5)


/tmp/ipykernel_1260427/1405874126.py:37: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)



In [5]:
with torch.no_grad():
    generated_p = model.sample(labels)

print(generated_p)

tensor([[[-0.8657, -0.9779, -0.8309, -0.7863, -0.8695, -0.7897, -0.9844,
          -0.9904, -0.8234, -1.0000, -0.8724, -0.7648, -0.7974, -0.9845,
          -0.9683, -0.7889, -0.7481, -0.8673, -0.7851, -0.7648, -0.8596,
          -0.7666, -0.7549, -0.9745, -0.7698, -0.9700, -0.9608, -0.7878,
          -0.9370, -0.8200, -0.7598, -0.9705, -0.7373, -0.9703, -0.7615,
          -0.8917, -0.9826, -0.8440, -0.7659, -0.7679, -0.8388, -0.8314,
          -0.9939, -0.7528, -0.9815, -0.8165, -0.9875, -0.9173, -0.9370,
          -0.8900, -0.8674, -0.7464, -0.7860, -0.8313, -0.7566, -0.7470,
          -0.7490, -0.8888, -0.9804, -0.9754, -0.9164, -0.8409,  0.1887,
          -0.9384, -0.8582, -0.9688, -0.8109, -0.9330, -0.9820, -0.7979,
          -0.9704, -0.9175, -0.7564, -0.9341, -0.9363, -0.9960, -0.7711,
          -0.9046, -0.9098, -0.9161, -0.9850, -0.9550, -0.8906, -0.9007,
          -0.9685, -0.7958,  0.0260, -0.1362, -0.8117, -0.9434, -0.1205,
           0.3678, -0.9304, -0.9388, -0.7586,  0.37

**Visualise the GAN-generated images:**

In [6]:
'''
Plot the generated images!
'''


#generated_p = generated_p.numpy()
# copy the image to a pure numpy array

print(generated_p.shape)
generated_p_1 = generated_p[0][0]

min_charge = 0
max_charge = np.log(test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max'] + 1)

generated_p_1 = (generated_p_1 + 1) / 2
generated_p_1 *= (max_charge - min_charge)
generated_p_1 += min_charge

generated_p_1 = np.exp(generated_p_1) - 1
print(generated_p_1.shape)
# reshape the generated image to a 5x5x5 array
generated_p_1 = generated_p_1.reshape(5, 5, 5)



generated_plot = np.zeros((5, 5, 5))

for i in range(generated_plot.shape[0]):
    for j in range(generated_plot.shape[1]):
        for k in range(generated_plot.shape[2]):
            generated_plot[i, j, k] = float(generated_p_1[i, j, k])
print(generated_plot)
# check the type of the elements in the array
print(generated_plot.dtype)

# Max deposited energy in one voxel
max_energy = generated_plot.max()

#generated_plot[generated_plot > 150] = 0
generated_plot[generated_plot < 0] = 0
print(max_energy)




torch.Size([1, 1, 125])
torch.Size([125])
[[[6.65045619e-01 8.75222683e-02 9.00587797e-01 1.25062180e+00
   6.41065121e-01]
  [1.22231030e+00 6.10595942e-02 3.70435715e-02 9.54931259e-01
   6.56843185e-05]
  [6.23464227e-01 1.44245934e+00 1.15812469e+00 6.04369640e-02
   1.27902627e-01]
  [1.22907925e+00 1.60224080e+00 6.54949546e-01 1.26103878e+00
   1.44280863e+00]
  [7.03808665e-01 1.42551303e+00 1.53604245e+00 1.01709485e-01
   1.39618492e+00]]

 [[1.20574117e-01 1.60362005e-01 1.23848724e+00 2.70063996e-01
   9.80306149e-01]
  [1.48918056e+00 1.18599534e-01 1.71150255e+00 1.19241834e-01
   1.47364521e+00]
  [5.08346915e-01 6.84244633e-02 8.08357239e-01 1.43210602e+00
   1.41394067e+00]
  [8.43968749e-01 8.96914721e-01 2.35244036e-02 1.55631304e+00
   7.28940964e-02]
  [1.00748158e+00 4.84431982e-02 3.68846536e-01 2.70035625e-01
   5.18226385e-01]]

 [[6.54153347e-01 1.61951447e+00 1.25310731e+00 8.97141933e-01
   1.51928139e+00]
  [1.61312652e+00 1.59316230e+00 5.25569797e-01 7.73

/tmp/ipykernel_1812279/2216004893.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  generated_p_1 = np.exp(generated_p_1) - 1


In [7]:

print("- Proton image:")
plotly_generate(generated_plot, max_energy=max_energy)


- Proton image:
184
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
90
1
1
1
1
1
48
25
1
27
179
1
183
1
1
1
1
1
1
1
1
1
1


In [8]:
event = test_set_p[300]
true_img = np.zeros((125,))

for i in range(true_img.shape[0]):
    true_img[i] = float(event["image"][i])

true_img = true_img.reshape(5, 5, 5)

# get back the normalization
min_charge = 0
max_charge = np.log(test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max'] + 1)
true_img = (true_img + 1) / 2
true_img *= (max_charge - min_charge)
true_img += min_charge
true_img = np.exp(true_img) - 1
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['std'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['mean'])

max_energy = true_img.max()

print(true_img)

print("- True image:")
plotly_generate(true_img, max_energy=max_energy)



303.3067761656923
197.1132585084357
[[[7.26919515e-02 1.02291458e+00 1.31553240e+00 4.28579112e-02
   5.69811849e-02]
  [5.52373395e-01 2.71233043e-01 1.08408569e+00 1.56950444e+00
   1.28656979e+00]
  [2.23761724e-01 1.46396125e+00 9.09114637e-01 9.42342422e-01
   5.84914101e-01]
  [2.72652549e-01 6.53120668e-01 9.71488117e-01 4.20262038e-01
   1.22679991e+00]
  [1.12380101e-01 6.79161524e-01 5.23724943e-01 2.86253106e-01
   8.73169245e-01]]

 [[1.19749060e+00 1.36853173e+00 9.51624994e-01 4.89308139e-01
   5.59974949e-01]
  [1.18118930e+00 1.02122218e+00 4.78496738e-01 5.79483247e-01
   4.11901157e-01]
  [1.34068617e+00 1.75475529e-01 6.32835623e-01 1.57198138e+00
   1.25850839e+00]
  [1.80244134e-01 1.33434434e-01 1.45543923e+00 8.07674877e-01
   7.48319876e-03]
  [1.01490771e-01 9.93682009e-02 1.35087509e+00 8.54556585e-02
   5.77078438e-01]]

 [[1.21307059e+00 1.45446997e-01 1.71061353e+00 5.32691316e-01
   2.24751435e-01]
  [1.16082442e+00 1.47372820e+00 8.89454935e-01 1.15989676

In [ ]:
print(event["image"].numpy())

[ 197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  266.32901016  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  261.34530654  197.11325851  197.11325851
  197.11325851  263.76361099 1348.25107589  298.85057845  197.11325851
  197.11325851  197.11325851  286.39696517  197.11325851  197.11325851
  197.